# Modrinth Project Versions — Bronze to Silver

This notebook reads the latest detailed Modrinth version payloads from Bronze, expands each project's version-array payload into one row per Modrinth version, normalizes nested values for Polars, and writes the inferred result to Silver.

It mirrors the refactored Modrinth General Bronze-to-Silver notebook: configuration comes from YAML, paths come from shared utilities, and no `Bundle` object is used.


In [ ]:
import os
import json

import duckdb
import polars as pl
import yaml
from IPython.display import display

from shared_code.database.directory_creation import *
from shared_code.database.database_and_schema_creation import *
from pipelines.modrinth_api_detail.schemas.modrinth_detail_schemas import *


In [ ]:
def read_latest_raw_project_versions(
    bronze_db_path: str,
    table_name: str,
) -> pl.DataFrame:
    """Read the latest Bronze project-version payloads for every project type."""
    query = f"""
        WITH latest_run AS (
            SELECT
                run_id,
                project_type
            FROM (
                SELECT
                    run_id,
                    project_type,
                    MAX(c_pull_timestamp_utc) AS max_pull_timestamp,
                    ROW_NUMBER() OVER (
                        PARTITION BY project_type
                        ORDER BY
                            MAX(c_pull_timestamp_utc) DESC,
                            run_id DESC
                    ) AS rn
                FROM {table_name}
                GROUP BY
                    run_id,
                    project_type
            )
            WHERE rn = 1
        )

        SELECT
            r.run_id,
            r.project_type,
            r.project_id,
            r.payload,
            r.c_pull_timestamp_utc
        FROM {table_name} r

        INNER JOIN latest_run l
            ON r.run_id = l.run_id
            AND r.project_type = l.project_type
    """

    with duckdb.connect(
        bronze_db_path,
        read_only=True,
    ) as bronze_con:
        bronze_con.execute(
            "SET arrow_large_buffer_size = true"
        )

        return bronze_con.execute(query).pl()


In [ ]:
def get_raw_project_version_counts(
    bronze_db_path: str,
    table_name: str,
) -> pl.DataFrame:
    """Return project-payload and nested-version counts for the latest run."""
    query = f"""
        WITH latest_run AS (
            SELECT
                project_type,
                run_id
            FROM (
                SELECT
                    project_type,
                    run_id,
                    MAX(c_pull_timestamp_utc) AS max_pull_timestamp,
                    ROW_NUMBER() OVER (
                        PARTITION BY project_type
                        ORDER BY
                            MAX(c_pull_timestamp_utc) DESC,
                            run_id DESC
                    ) AS row_num
                FROM {table_name}
                GROUP BY
                    project_type,
                    run_id
            )
            WHERE row_num = 1
        )

        SELECT
            r.run_id,
            r.project_type,
            COUNT(*)::BIGINT AS project_payloads,
            SUM(
                json_array_length(r.payload)
            )::BIGINT AS versions_fetched
        FROM {table_name} r

        INNER JOIN latest_run l
            ON r.run_id = l.run_id
            AND r.project_type = l.project_type

        GROUP BY
            r.run_id,
            r.project_type

        ORDER BY
            r.project_type
    """

    with duckdb.connect(
        bronze_db_path,
        read_only=True,
    ) as bronze_con:
        return bronze_con.execute(query).pl()


In [ ]:
def normalize_version_value(value):
    """Normalize API values before Polars schema inference."""
    if isinstance(value, dict):
        return json.dumps(value)

    if isinstance(value, list):
        # Empty arrays have no inferable inner dtype.
        if not value:
            return None

        # Complex nested arrays such as files/dependencies remain JSON.
        contains_nested_values = any(
            isinstance(item, (dict, list))
            for item in value
        )

        if contains_nested_values:
            return json.dumps(value)

        # Simple arrays such as loaders/game_versions remain native lists.
        return value

    return value


def unpack_project_versions(
    raw_df: pl.DataFrame,
) -> pl.DataFrame:
    """
    Expand each project's version payload into one row per Modrinth version.

    Bronze grain:
        one row per run_id + project_type + project_id

    Silver grain:
        one row per Modrinth version
    """
    version_rows = []

    metadata_columns = [
        column
        for column in raw_df.columns
        if column != "payload"
    ]

    for row in raw_df.iter_rows(named=True):
        metadata = {
            column: row[column]
            for column in metadata_columns
        }

        versions = json.loads(
            row["payload"]
        )

        if not versions:
            continue

        for version in versions:
            version_data = {
                key: normalize_version_value(value)
                for key, value in version.items()
            }

            # Bronze envelope metadata stays authoritative.
            for metadata_column in metadata_columns:
                version_data.pop(
                    metadata_column,
                    None,
                )

            version_rows.append(
                {
                    **version_data,
                    **metadata,
                }
            )

    if not version_rows:
        return pl.DataFrame()

    return pl.from_dicts(
        version_rows,
        infer_schema_length=None,
        strict=False,
    )


In [ ]:
def write_polars_to_silver(
    silver_db_path: str,
    df: pl.DataFrame,
    table_name: str,
) -> None:
    """Create or replace the Silver table using the Polars-inferred schema."""
    with duckdb.connect(silver_db_path) as silver_con:
        silver_con.register(
            "source_df",
            df,
        )

        silver_con.execute(
            f"""
            CREATE OR REPLACE TABLE {table_name} AS
            SELECT *
            FROM source_df
            """
        )

        silver_con.unregister(
            "source_df"
        )


In [ ]:
def create_base_project_versions(
    bronze_db_path: str,
    silver_db_path: str,
    table_name: str,
) -> None:
    raw_df = read_latest_raw_project_versions(
        bronze_db_path=bronze_db_path,
        table_name=table_name,
    )

    print(
        f"Read in {raw_df.height:,} "
        "project version payloads from bronze."
    )

    raw_count_df = get_raw_project_version_counts(
        bronze_db_path=bronze_db_path,
        table_name=table_name,
    )

    display(raw_count_df)

    silver_df = unpack_project_versions(
        raw_df
    )

    print(
        f"Expanded into {silver_df.height:,} version rows."
    )

    write_polars_to_silver(
        silver_db_path=silver_db_path,
        df=silver_df,
        table_name=table_name,
    )

    print(
        f"Silver project versions written: "
        f"{silver_df.height:,}"
    )


In [ ]:
config_path = "stream-config.yml"

with open(config_path, "r") as file:
    config_data = yaml.safe_load(file)


In [ ]:
env = "dev"

project_root = os.path.abspath(
    os.path.join(
        os.getcwd(),
        "..",
        "..",
        "..",
    )
)

streams = config_data["streams"]

detail_stream = next(
    (
        stream
        for stream in streams
        if stream["name"] == "modrinth_project_versions"
    ),
    None,
)

if detail_stream is None:
    raise ValueError(
        "stream-config.yml must contain a stream named "
        "'modrinth_project_versions'."
    )

source_url = detail_stream["source-url"]
table_name = detail_stream["table-name"]

bronze_path = os.path.join(
    project_root,
    config_data["main"]["parent-folder"],
    "bronze",
    env,
    build_db_filename(source_url),
)

silver_path = os.path.join(
    project_root,
    config_data["main"]["parent-folder"],
    "silver",
    env,
    build_db_filename(source_url),
)

print(bronze_path)
print(silver_path)


In [ ]:
# Silver is inferred from the Polars dataframe, matching Modrinth General.
build_layer_directory(
    os.path.dirname(silver_path)
)

init_db(
    db_path=silver_path
)

create_base_project_versions(
    bronze_db_path=bronze_path,
    silver_db_path=silver_path,
    table_name=table_name,
)


In [ ]:
with duckdb.connect(
    silver_path,
    read_only=True,
) as silver_con:
    silver_df = silver_con.execute(
        f"""
        SELECT *
        FROM {table_name}
        """
    ).pl()

with pl.Config(
    tbl_rows=5,
    tbl_cols=35,
):
    display(silver_df)
